In [3]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

BASE_DIR = "data"
target_col = "fluency_mean_2ann"   # human average
predicted_col = "fluency_score"    # LLM judge

results = []
for filepath in sorted(glob.glob(os.path.join(BASE_DIR, "**", "*.csv"), recursive=True)):
    df = pd.read_csv(filepath)
    if target_col not in df.columns or predicted_col not in df.columns:
        continue

    valid = df[[target_col, predicted_col]].dropna()
    if valid.empty:
        continue

    y_true = valid[target_col].astype(float)
    y_pred = valid[predicted_col].astype(float)

    results.append({
        "file": os.path.relpath(filepath, BASE_DIR),
        "n_rows": len(valid),
        "Spearman": spearmanr(y_true, y_pred).statistic
                    if y_true.nunique() > 1 and y_pred.nunique() > 1 else np.nan,
        "MAE": mean_absolute_error(y_true, y_pred),
        "bias": float(np.mean(y_pred - y_true)),   # positive = LLM scores higher than humans
        "within_1.0": float(np.mean(np.abs(y_true - y_pred) <= 1.0)),
    })

results_df = pd.DataFrame(results)
print(results_df.round(3).to_string(index=False))

                                       file  n_rows  Spearman   MAE   bias  within_1.0
   annotated_rows_with_fluency_acehnese.csv     810     0.694 0.864  0.577       0.753
       annotated_rows_with_fluency_hitu.csv     810     0.152 1.699  1.188       0.372
 annotated_rows_with_fluency_indonesian.csv     810     0.369 0.117 -0.041       0.980
   annotated_rows_with_fluency_javanese.csv     810     0.343 0.872  0.444       0.756
annotated_rows_with_fluency_minangkabau.csv     810     0.468 0.914  0.599       0.719
 annotated_rows_with_fluency_vietnamese.csv     810     0.391 0.436  0.403       0.888
